In [0]:
%sql
use catalog identifier(:target_catalog)

In [0]:
%sql

create or replace temporary view policy_start as
select
  policy_id,
  member_id,
  min(__START_AT) as policy_start_date
from silver.policy_history
group by policy_id, member_id;

create or replace temporary view network_membership as
select
  c.claim_id,
  max(case when nh.hospital_id is not null then true else false end) as is_network_at_admission
from silver.claims_clean c
left join silver.hospital_network_history_clean nh
  on nh.hospital_id = c.hospital_id
 and to_date(nh.network_from) <= to_date(c.admission_date)
 and (nh.network_to is null or to_date(nh.network_to) >= to_date(c.admission_date))
group by c.claim_id;

create or replace table gold.claims_enriched
cluster by (admission_month, hospital_id)
comment 'Member claim facts enriched to policy version at admission, plan attributes and hospital network status. Published as a Delta table by the publish_gold job task; state-scoped by the gov.claims_state_filter row filter on hospital_id.'
as
select
  c.claim_id,
  c.member_id,
  c.policy_id,
  c.hospital_id,
  c.admission_date,
  c.discharge_date,
  trunc(to_date(c.admission_date), 'MONTH') as admission_month,
  c.diagnosis_code,
  c.procedure_code,
  c.is_pre_existing,
  c.claim_type,
  c.billed_amount,
  c.room_rent_per_day,
  c.room_days,
  c.status,
  c.approved_amount,
  c.rejection_code,
  c.submitted_ts,
  c.settled_ts,
  c.channel,
  ph.plan_code as plan_code_at_admission,
  p.plan_family,
  p.room_rent_cap_per_day,
  ps.policy_start_date,
  cast(months_between(to_date(c.admission_date), ps.policy_start_date) as int) as policy_tenure_months,
  cast(datediff(to_date(c.discharge_date), to_date(c.admission_date)) as int) as length_of_stay,
  case
    when c.settled_ts is not null and c.submitted_ts is not null
      then datediff(cast(c.settled_ts as timestamp), cast(c.submitted_ts as timestamp))
  end as turnaround_days,
  c.room_rent_per_day > try_cast(p.room_rent_cap_per_day as double) as room_rent_over_cap,
  nm.is_network_at_admission,
  h.city as hospital_city,
  h.city_tier as hospital_city_tier
from silver.claims_clean c
left join silver.policy_history ph
  on ph.policy_id = c.policy_id
 and ph.member_id = c.member_id
 and ph.__START_AT <= to_date(c.admission_date)
 and (ph.__END_AT is null or ph.__END_AT > to_date(c.admission_date))
left join silver.plans_clean p
  on p.plan_code = upper(ph.plan_code)
left join silver.hospitals_clean h
  on h.hospital_id = c.hospital_id
left join policy_start ps
  on ps.policy_id = c.policy_id
 and ps.member_id = c.member_id
left join network_membership nm
  on nm.claim_id = c.claim_id;

-- R1 governance: re-apply the state-scoped row filter (lost when the MV was dropped;
-- on a table it is ALTER TABLE instead of ALTER MATERIALIZED VIEW).


In [0]:
%sql

create or replace table gold.hospital_monthly
comment 'Per hospital × admission month aggregates (claim_count, total_billed, total_approved, rejected_count) over the published claims_enriched table. Published by the publish_gold job task.'
as
select
  hospital_id,
  admission_month,
  cast(count(*) as int) as claim_count,
  cast(sum(billed_amount) as decimal(18,2)) as total_billed,
  cast(coalesce(sum(approved_amount), 0) as decimal(18,2)) as total_approved,
  cast(sum(case when status = 'rejected' then 1 else 0 end) as int) as rejected_count
from gold.claims_enriched
group by hospital_id, admission_month;


In [0]:
%sql

select 'claims_enriched'  t, cast(count(*) as bigint) n, cast(count(distinct claim_id) as bigint) d
from gold.claims_enriched
union all
select 'hospital_monthly', cast(count(*) as bigint), cast(sum(claim_count) as bigint)
from gold.hospital_monthly;

select table_catalog, table_schema, table_name, filter_name, target_columns
from information_schema.row_filters
where table_schema = 'gold' and table_name = 'claims_enriched';